# 🔥 Firefighter Mortality Analysis

## Objective

This project explores patterns in firefighter fatalities, focusing on the relationship between **rank, age, and category (firefighters vs officers)**.

The goal is to investigate whether **lower-ranking firefighters tend to die more frequently or at younger ages compared to higher-ranking officers**.

This project was developed as part of my learning journey in **Data Science and Data Analysis using Python**.

---
**Dataset:** 2,005 records of U.S. firefighter fatalities  
**Final sample (after cleaning):** 1,645 valid records  
**Period covered:** 2000–present

## Technologies Used

- Python 3
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Jupyter Notebook

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

# ── Professional theme ──────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
PALETTE_CAT  = ["#E63946", "#457B9D"]          # Firefighters / Officers
PALETTE_RANK = sns.color_palette("rocket_r", 12)
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
})

## 1. Load Dataset
The dataset contains records of firefighter fatalities including rank, age, cause of death and incident information.

In [ ]:
df = pd.read_csv("database.csv")
print(f"Raw dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Initial Data Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning
Remove irrelevant or personally-identifiable columns and standardize the Rank field.

In [ ]:
columns_to_remove = [
    "Unnamed: 13",
    "First Name",
    "Last Name",
    "Activity",
    "Emergency",
    "Property Type",
]

df = df.drop(columns=columns_to_remove, errors="ignore")

df["Rank"] = (
    df["Rank"]
    .astype(str)
    .str.strip()
    .str.title()
)

## 4. Age Cleaning
Convert the `Age` column to numeric and drop rows with missing age values.

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")

missing_age = df["Age"].isna().sum()
print(f"Records dropped due to missing Age: {missing_age}")

df = df.dropna(subset=["Age"])
print(f"Clean dataset: {len(df):,} records")

## 5. Distribution of Deaths by Rank
Which ranks account for the highest number of fatalities?

In [ ]:
rank_order = df["Rank"].value_counts().index

fig, ax = plt.subplots(figsize=(12, 8))

sns.countplot(
    data=df,
    y="Rank",
    order=rank_order,
    palette="rocket_r",
    ax=ax,
)

# Annotate each bar with count
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(
            f"{int(width):,}",
            xy=(width, p.get_y() + p.get_height() / 2),
            xytext=(4, 0),
            textcoords="offset points",
            va="center",
            fontsize=8,
            color="#333333",
        )

ax.set_title("Number of Firefighter Fatalities by Rank", pad=15)
ax.set_xlabel("Number of Deaths", labelpad=10)
ax.set_ylabel("Rank", labelpad=10)
ax.xaxis.set_major_locator(ticker.MultipleLocator(50))

plt.tight_layout()
plt.savefig("plot_deaths_by_rank.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Age Distribution by Rank
How does age at time of death vary across the most frequent ranks? (Only ranks with ≥ 10 records are shown individually.)

In [ ]:
rank_counts = df["Rank"].value_counts()

df["Rank_Grouped"] = df["Rank"].apply(
    lambda x: x if rank_counts[x] >= 10 else "Other"
)

In [ ]:
# Order boxes by median age for readability
grouped_order = (
    df.groupby("Rank_Grouped")["Age"]
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=(14, 6))

sns.boxplot(
    data=df,
    x="Rank_Grouped",
    y="Age",
    order=grouped_order,
    palette="coolwarm",
    linewidth=1.2,
    flierprops=dict(marker="o", markersize=3, alpha=0.4),
    ax=ax,
)

ax.set_title("Age at Death — Distribution by Rank", pad=15)
ax.set_xlabel("Rank", labelpad=10)
ax.set_ylabel("Age", labelpad=10)
ax.tick_params(axis="x", rotation=40)

plt.tight_layout()
plt.savefig("plot_age_by_rank.png", dpi=150, bbox_inches="tight")
plt.show()

## 7. Firefighters vs Officers
Ranks are grouped into two operational categories to enable a higher-level comparison.

In [ ]:
def classify_rank(rank: str) -> str:
    """Return 'Officers' if the rank contains an officer-level keyword, else 'Firefighters'."""
    officer_keywords = [
        "Lieutenant",
        "Captain",
        "Chief",
        "Battalion Chief",
        "Assistant Chief",
        "Deputy Chief",
    ]
    return "Officers" if any(kw in rank for kw in officer_keywords) else "Firefighters"


df["Category"] = df["Rank"].apply(classify_rank)

### 7.1 Number of Deaths by Category

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

counts = df["Category"].value_counts()

bars = sns.countplot(
    data=df,
    x="Category",
    order=counts.index,
    palette=PALETTE_CAT,
    ax=ax,
)

# Percentage labels
total = len(df)
for p in ax.patches:
    h = p.get_height()
    ax.text(
        p.get_x() + p.get_width() / 2,
        h + 10,
        f"{int(h):,}\n({h/total:.1%})",
        ha="center",
        fontsize=11,
        fontweight="bold",
    )

ax.set_title("Fatalities: Firefighters vs Officers", pad=15)
ax.set_xlabel("Category", labelpad=10)
ax.set_ylabel("Number of Deaths", labelpad=10)
ax.set_ylim(0, counts.max() * 1.15)

plt.tight_layout()
plt.savefig("plot_category_count.png", dpi=150, bbox_inches="tight")
plt.show()

### 7.2 Age Distribution by Category

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ── Box plot ─────────────────────────────────────────────────────────────
sns.boxplot(
    data=df,
    x="Category",
    y="Age",
    palette=PALETTE_CAT,
    linewidth=1.4,
    flierprops=dict(marker="o", markersize=3, alpha=0.35),
    ax=axes[0],
)
axes[0].set_title("Age Distribution by Category")
axes[0].set_xlabel("Category")
axes[0].set_ylabel("Age at Death")

# ── KDE / Histogram overlay ───────────────────────────────────────────────
for cat, color in zip(["Firefighters", "Officers"], PALETTE_CAT):
    subset = df.loc[df["Category"] == cat, "Age"]
    axes[1].hist(
        subset,
        bins=20,
        alpha=0.45,
        color=color,
        label=f"{cat} (n={len(subset):,})",
        edgecolor="white",
    )
    axes[1].axvline(subset.mean(), color=color, linestyle="--", linewidth=1.5)

axes[1].set_title("Age at Death — Frequency Distribution")
axes[1].set_xlabel("Age")
axes[1].set_ylabel("Count")
axes[1].legend()

plt.suptitle("Firefighters vs Officers — Age Comparison", fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("plot_age_category.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Summary Statistics

In [ ]:
# ── Complete descriptive statistics grouped by category ────────────────────
age_stats = df.groupby("Category")["Age"].agg(
    Count="count",
    Mean="mean",
    Median="median",
    Std="std",
    Min="min",
    Max="max",
    Q1=lambda x: x.quantile(0.25),
    Q3=lambda x: x.quantile(0.75),
).round(2)

print("=== Age Statistics by Category ===")
display(age_stats)

In [ ]:
# ── Value counts and share by category ────────────────────────────────────
category_counts = df["Category"].value_counts().rename("Count")
category_pct    = (df["Category"].value_counts(normalize=True) * 100).round(2).rename("Share (%)")

print("=== Fatality Count and Share by Category ===")
display(pd.concat([category_counts, category_pct], axis=1))

print("\n=== Top 10 Ranks by Fatality Count ===")
display(
    df["Rank"].value_counts()
    .head(10)
    .rename_axis("Rank")
    .reset_index(name="Count")
)

In [ ]:
# ── Cause of Death breakdown ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cause_order = df["Cause Of Death"].value_counts().index

sns.countplot(
    data=df,
    y="Cause Of Death",
    order=cause_order,
    palette="flare",
    ax=axes[0],
)
axes[0].set_title("Fatalities by Cause of Death")
axes[0].set_xlabel("Count")
axes[0].set_ylabel("Cause")

nature_order = df["Nature Of Death"].value_counts().index
sns.countplot(
    data=df,
    y="Nature Of Death",
    order=nature_order,
    palette="crest",
    ax=axes[1],
)
axes[1].set_title("Fatalities by Nature of Death")
axes[1].set_xlabel("Count")
axes[1].set_ylabel("Nature")

plt.tight_layout()
plt.savefig("plot_cause_nature.png", dpi=150, bbox_inches="tight")
plt.show()

---

## 9. Conclusion

### Does rank correlate with age at death or mortality frequency?

The analysis of **1,645 valid firefighter fatality records** reveals a statistically meaningful pattern between operational category and both mortality frequency and age at time of death.

---

#### 9.1 Mortality Frequency

| Category     | Count | Share  |
|:-------------|------:|-------:|
| Firefighters | 1,054 | 64.07% |
| Officers     |   591 | 35.93% |

Operational-level firefighters account for nearly **two-thirds of all fatalities** in the dataset. The rank *Firefighter* alone represents **663 records** — more than four times the next most frequent rank (*Captain*, 148). This gap is consistent with the higher operational exposure of front-line personnel.

---

#### 9.2 Age at Death

| Category     | Mean Age | Median Age | Std Dev | Min | Max |
|:-------------|:--------:|:----------:|:-------:|:---:|:---:|
| Firefighters | 43.97    | 44.0       | 15.45   | 14  | 91  |
| Officers     | 50.53    | 50.0       | 11.38   | 19  | 95  |

Officers die, on average, **~6.6 years older** than operational firefighters. Additionally, the standard deviation for Firefighters (15.45) is considerably higher than for Officers (11.38), indicating greater age dispersion — including a non-trivial number of very young fatalities at the Firefighter level.

---

#### 9.3 Principal Causes

The leading cause of death across both categories is **Stress/Overexertion (831 cases, ~50.5%)**, driven primarily by **heart attacks (766 records)**. This pattern affects officers disproportionately given their older age profile. Trauma-based causes (vehicle collisions, impact, entrapment) are more prevalent in younger firefighters.

---

#### 9.4 Final Assessment

> **The data supports the hypothesis that lower-ranking firefighters face a higher mortality burden, both in absolute frequency and at younger ages.**
>
> This pattern is consistent with greater physical and situational risk exposure for front-line personnel. Officers, while dying at older ages, are not exempt — heart-attack-related fatalities remain prevalent across all levels.
>
> A formal confirmation of statistical significance (e.g., Mann-Whitney U test on age distributions) would be a recommended next step before drawing causal conclusions.

---

*Analysis conducted with Pandas, Matplotlib and Seaborn. Dataset sourced from U.S. firefighter fatality records.*